# Exploratory Data Analysis (EDA) - Telco Customer Churn

This notebook performs exploratory data analysis on the Telco Customer Churn dataset to understand patterns and factors influencing customer churn.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print('Libraries imported successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Dataset shape: {df.shape}')
print(f'Number of rows: {df.shape[0]}')
print(f'Number of columns: {df.shape[1]}')

## 1. Data Overview

In [ ]:
# Display first few rows
df.head(10)

In [ ]:
# Data types and info
df.info()

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage
})
print('Missing Values:')
missing_df[missing_df['Missing Values'] > 0]

In [ ]:
# Check for empty string values in TotalCharges (which may represent missing data)
empty_total_charges = df[df['TotalCharges'].str.strip() == ''].shape[0]
print(f'Rows with empty TotalCharges: {empty_total_charges}')

# Also check for whitespace-only entries in other string columns
for col in df.select_dtypes(include=['object']).columns:
    empty_count = df[df[col].str.strip() == ''].shape[0]
    if empty_count > 0:
        print(f"Column '{col}' has {empty_count} empty string values")

In [ ]:
# Statistical summary - numerical columns
df[['tenure', 'MonthlyCharges', 'TotalCharges']].describe()

In [ ]:
# Convert TotalCharges to numeric properly
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Fill NaN TotalCharges with 0 (these are new customers with no charges yet)
df['TotalCharges'] = df['TotalCharges'].fillna(0)

print('TotalCharges converted to numeric successfully.')
print(f'Unique customers with TotalCharges = 0: {(df["TotalCharges"] == 0).sum()}')

## 2. Target Variable Analysis - Churn Distribution

In [ ]:
# Churn distribution
churn_counts = df['Churn'].value_counts()
churn_percentages = df['Churn'].value_counts(normalize=True) * 100

print('Churn Distribution:')
print(f"No Churn: {churn_counts['No']} ({churn_percentages['No']:.2f}%)")
print(f"Churn:    {churn_counts['Yes']} ({churn_percentages['Yes']:.2f}%)")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
colors = ['#4CAF50', '#FF5722']
axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Churn Distribution (Count)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold', fontsize=12)

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0.05, 0.05))
axes[1].set_title('Churn Distribution (Percentage)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../images/churn_distribution_eda.png', dpi=100, bbox_inches='tight')
plt.show()

## 3. Customer Demographics Analysis

In [ ]:
# Gender analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gender distribution
gender_counts = df['gender'].value_counts()
axes[0].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
            colors=['#66B2FF', '#FF9999'], startangle=90)
axes[0].set_title('Gender Distribution', fontsize=14, fontweight='bold')

# Churn rate by gender
gender_churn = pd.crosstab(df['gender'], df['Churn'], normalize='index') * 100
gender_churn.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF5722'])
axes[1].set_title('Churn Rate by Gender', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Gender')
axes[1].legend(title='Churn')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Senior Citizen analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Senior Citizen distribution
senior_counts = df['SeniorCitizen'].value_counts().sort_index()
labels = ['Non-Senior (0)', 'Senior (1)']
axes[0].pie(senior_counts.values, labels=labels, autopct='%1.1f%%',
            colors=['#66B2FF', '#FF9999'], startangle=90)
axes[0].set_title('Senior Citizen Distribution', fontsize=14, fontweight='bold')

# Churn rate by Senior Citizen
senior_churn = pd.crosstab(df['SeniorCitizen'], df['Churn'], normalize='index') * 100
senior_churn.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF5722'])
axes[1].set_title('Churn Rate by Senior Citizen Status', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Senior Citizen')
axes[1].legend(title='Churn')
axes[1].set_xticklabels(labels, rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Partner and Dependents analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Partner
partner_churn = pd.crosstab(df['Partner'], df['Churn'], normalize='index') * 100
partner_churn.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#FF5722'])
axes[0].set_title('Churn Rate by Partner Status', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_xlabel('Has Partner')
axes[0].legend(title='Churn')
axes[0].tick_params(axis='x', rotation=0)

# Dependents
dependents_churn = pd.crosstab(df['Dependents'], df['Churn'], normalize='index') * 100
dependents_churn.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF5722'])
axes[1].set_title('Churn Rate by Dependents Status', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].set_xlabel('Has Dependents')
axes[1].legend(title='Churn')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## 4. Services Analysis

In [ ]:
# Phone Service
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Phone Service
phone_churn = pd.crosstab(df['PhoneService'], df['Churn'], normalize='index') * 100
phone_churn.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#FF5722'])
axes[0].set_title('Churn Rate by Phone Service', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Churn')

# Multiple Lines
mlines_churn = pd.crosstab(df['MultipleLines'], df['Churn'], normalize='index') * 100
mlines_churn.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF5722'])
axes[1].set_title('Churn Rate by Multiple Lines', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Churn')

# Internet Service
inet_churn = pd.crosstab(df['InternetService'], df['Churn'], normalize='index') * 100
inet_churn.plot(kind='bar', ax=axes[2], color=['#4CAF50', '#FF5722'])
axes[2].set_title('Churn Rate by Internet Service', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Percentage (%)')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Online Security, Backup, Device Protection, Tech Support
service_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport']

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    service_churn = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    service_churn.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#FF5722'])
    axes[i].set_title(f'Churn Rate by {col}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].legend(title='Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Streaming TV and Movies
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stream_cols = ['StreamingTV', 'StreamingMovies']
for i, col in enumerate(stream_cols):
    stream_churn = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    stream_churn.plot(kind='bar', ax=axes[i], color=['#4CAF50', '#FF5722'])
    axes[i].set_title(f'Churn Rate by {col}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Percentage (%)')
    axes[i].tick_params(axis='x', rotation=15)
    axes[i].legend(title='Churn')

plt.tight_layout()
plt.show()

## 5. Contract and Billing Analysis

In [ ]:
# Contract Type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contract distribution
contract_counts = df['Contract'].value_counts()
axes[0].pie(contract_counts.values, labels=contract_counts.index, autopct='%1.1f%%',
            colors=['#FF9999', '#66B2FF', '#99FF99'], startangle=90)
axes[0].set_title('Contract Type Distribution', fontsize=14, fontweight='bold')

# Churn rate by contract
contract_churn = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
contract_churn.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF5722'])
axes[1].set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Churn')

plt.tight_layout()
plt.show()

In [ ]:
# Paperless Billing
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

paperless_churn = pd.crosstab(df['PaperlessBilling'], df['Churn'], normalize='index') * 100
paperless_churn.plot(kind='bar', ax=axes[0], color=['#4CAF50', '#FF5722'])
axes[0].set_title('Churn Rate by Paperless Billing', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Percentage (%)')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='Churn')

# Payment Method
payment_churn = pd.crosstab(df['PaymentMethod'], df['Churn'], normalize='index') * 100
payment_churn.plot(kind='bar', ax=axes[1], color=['#4CAF50', '#FF5722'])
axes[1].set_title('Churn Rate by Payment Method', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Churn')

plt.tight_layout()
plt.show()

## 6. Numerical Analysis - Tenure & Charges

In [ ]:
# Tenure distribution by churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
churned = df[df['Churn'] == 'Yes']['tenure']
not_churned = df[df['Churn'] == 'No']['tenure']

axes[0].hist([not_churned, churned], bins=30, alpha=0.7,
             label=['No Churn', 'Churn'], color=['#4CAF50', '#FF5722'], edgecolor='black')
axes[0].set_title('Tenure Distribution by Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Box plot
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[1], palette=['#4CAF50', '#FF5722'])
axes[1].set_title('Tenure Distribution by Churn (Box Plot)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Tenure (months)')

plt.tight_layout()
plt.show()

# Print summary statistics
print('\nTenure Statistics:')
print(f"No Churn - Mean: {not_churned.mean():.2f}, Median: {not_churned.median():.2f}, Std: {not_churned.std():.2f}")
print(f"Churn    - Mean: {churned.mean():.2f}, Median: {churned.median():.2f}, Std: {churned.std():.2f}")

In [ ]:
# Monthly Charges analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churned_mc = df[df['Churn'] == 'Yes']['MonthlyCharges']
not_churned_mc = df[df['Churn'] == 'No']['MonthlyCharges']

axes[0].hist([not_churned_mc, churned_mc], bins=30, alpha=0.7,
             label=['No Churn', 'Churn'], color=['#4CAF50', '#FF5722'], edgecolor='black')
axes[0].set_title('Monthly Charges Distribution by Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[1], palette=['#4CAF50', '#FF5722'])
axes[1].set_title('Monthly Charges by Churn (Box Plot)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

print('\nMonthly Charges Statistics:')
print(f"No Churn - Mean: ${not_churned_mc.mean():.2f}, Median: ${not_churned_mc.median():.2f}")
print(f"Churn    - Mean: ${churned_mc.mean():.2f}, Median: ${churned_mc.median():.2f}")

In [ ]:
# Total Charges analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

churned_tc = df[df['Churn'] == 'Yes']['TotalCharges']
not_churned_tc = df[df['Churn'] == 'No']['TotalCharges']

axes[0].hist([not_churned_tc, churned_tc], bins=30, alpha=0.7,
             label=['No Churn', 'Churn'], color=['#4CAF50', '#FF5722'], edgecolor='black')
axes[0].set_title('Total Charges Distribution by Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Total Charges ($)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

sns.boxplot(x='Churn', y='TotalCharges', data=df, ax=axes[1], palette=['#4CAF50', '#FF5722'])
axes[1].set_title('Total Charges by Churn (Box Plot)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Total Charges ($)')

plt.tight_layout()
plt.show()

print('\nTotal Charges Statistics:')
print(f"No Churn - Mean: ${not_churned_tc.mean():.2f}, Median: ${not_churned_tc.median():.2f}")
print(f"Churn    - Mean: ${churned_tc.mean():.2f}, Median: ${churned_tc.median():.2f}")

## 7. Correlation Analysis

In [ ]:
# Encode categorical variables for correlation
from sklearn.preprocessing import LabelEncoder

df_encoded = df.copy()

# Drop customerID
if 'customerID' in df_encoded.columns:
    df_encoded = df_encoded.drop('customerID', axis=1)

# Encode binary columns
binary_cols = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']
for col in binary_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Encode other categorical columns
cat_cols = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
            'Contract', 'PaymentMethod']
for col in cat_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])

# Encode target
df_encoded['Churn'] = df_encoded['Churn'].map({'Yes': 1, 'No': 0})

print('Data encoded for correlation analysis.')
df_encoded.head()

In [ ]:
# Correlation matrix
plt.figure(figsize=(16, 12))
correlation_matrix = df_encoded.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of All Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../images/correlation_matrix.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Top features correlated with Churn
churn_corr = correlation_matrix['Churn'].drop('Churn').sort_values(ascending=False)

plt.figure(figsize=(10, 8))
colors = ['#FF5722' if v > 0 else '#4CAF50' for v in churn_corr.values]
churn_corr.plot(kind='bar', color=colors, edgecolor='black')
plt.title('Feature Correlation with Churn', fontsize=14, fontweight='bold')
plt.xlabel('Features')
plt.ylabel('Correlation with Churn')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../images/churn_correlation.png', dpi=100, bbox_inches='tight')
plt.show()

print('\nTop 5 Positive Correlations with Churn:')
print(churn_corr.head(5))
print('\nTop 5 Negative Correlations with Churn:')
print(churn_corr.tail(5))

## 8. Tenure vs Monthly Charges Analysis

In [ ]:
# Scatter plot: Tenure vs Monthly Charges colored by Churn
plt.figure(figsize=(12, 7))

scatter = plt.scatter(
    df[df['Churn'] == 'No']['tenure'],
    df[df['Churn'] == 'No']['MonthlyCharges'],
    c='#4CAF50', alpha=0.5, label='No Churn', s=50, edgecolors='none'
)
scatter = plt.scatter(
    df[df['Churn'] == 'Yes']['tenure'],
    df[df['Churn'] == 'Yes']['MonthlyCharges'],
    c='#FF5722', alpha=0.5, label='Churn', s=50, edgecolors='none'
)

plt.title('Tenure vs Monthly Charges by Churn Status', fontsize=14, fontweight='bold')
plt.xlabel('Tenure (months)')
plt.ylabel('Monthly Charges ($)')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../images/tenure_vs_monthly_charges.png', dpi=100, bbox_inches='tight')
plt.show()

## 9. Key Insights Summary

Based on the EDA, here are the key findings:

1. **Class Imbalance**: The dataset has about 73.5% non-churn and 26.5% churn customers.

2. **Tenure**: Customers who churn have significantly lower tenure (average ~18 months) compared to non-churn customers (average ~38 months).

3. **Contract Type**: Month-to-month contracts have the highest churn rate. Two-year contracts have the lowest.

4. **Internet Service**: Fiber optic customers have a higher churn rate compared to DSL customers.

5. **Online Security & Tech Support**: Customers without online security or tech support are more likely to churn.

6. **Payment Method**: Electronic check users have the highest churn rate.

7. **Senior Citizens**: Senior citizens have a higher churn rate than non-seniors.

8. **Monthly Charges**: Churned customers tend to have higher monthly charges on average.